In [ ]:
# ============================================================
# 🖼️ IMAGE EDA - Comprehensive Image Analysis
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps, ImageFilter, ImageStat
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm import tqdm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Setup paths
train_images_path = "data/train/train"
train_csv_path = "data/train/train.csv"

# Load dataset
train_df = pd.read_csv(train_csv_path)
print(f"Total images to analyze: {len(train_df)}")

# ============================================================
# STEP 1: Image Metadata Collection
# ============================================================
print("\n" + "="*70)
print("STEP 1: Collecting Image Metadata...")
print("="*70)

image_data = []
failed_images = []

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Scanning images"):
    img_id = row['ImgId']
    img_path = os.path.join(train_images_path, f"{img_id}.jpg")
    
    try:
        if os.path.exists(img_path):
            img = Image.open(img_path).convert('RGB')
            file_size_kb = os.path.getsize(img_path) / 1024
            width, height = img.size
            aspect_ratio = width / height if height > 0 else 0
            
            # Color space analysis
            arr = np.array(img)
            r_mean = np.mean(arr[:, :, 0])
            g_mean = np.mean(arr[:, :, 1])
            b_mean = np.mean(arr[:, :, 2])
            
            # Quality metrics
            brightness = np.mean(arr)
            contrast = np.std(arr)
            
            # Sharpness (Laplacian variance)
            try:
                gray_img = img.convert('L')
                laplacian = gray_img.filter(ImageFilter.LAPLACIAN)
                sharpness = np.var(np.array(laplacian))
            except:
                sharpness = 0
            
            image_data.append({
                'ImgId': img_id,
                'width': width,
                'height': height,
                'file_size_kb': file_size_kb,
                'aspect_ratio': aspect_ratio,
                'r_mean': r_mean,
                'g_mean': g_mean,
                'b_mean': b_mean,
                'brightness': brightness,
                'contrast': contrast,
                'sharpness': sharpness,
                'category': row['categories']
            })
        else:
            failed_images.append(img_id)
    except Exception as e:
        failed_images.append(img_id)

# Create image statistics dataframe
image_stats_df = pd.DataFrame(image_data)
print(f"\n✓ Successfully analyzed {len(image_stats_df)} images")
print(f"⚠ Failed to analyze {len(failed_images)} images")

# Display basic statistics
print("\n" + "="*70)
print("IMAGE STATISTICS SUMMARY")
print("="*70)
print(f"\nDimensions:")
print(f"  • Width  - Mean: {image_stats_df['width'].mean():.0f}px, Std: {image_stats_df['width'].std():.0f}px")
print(f"  • Height - Mean: {image_stats_df['height'].mean():.0f}px, Std: {image_stats_df['height'].std():.0f}px")
print(f"  • Range: {image_stats_df['width'].min()}×{image_stats_df['height'].min()} to {image_stats_df['width'].max()}×{image_stats_df['height'].max()}")

print(f"\nFile Sizes:")
print(f"  • Mean: {image_stats_df['file_size_kb'].mean():.1f} KB")
print(f"  • Median: {image_stats_df['file_size_kb'].median():.1f} KB")
print(f"  • Range: {image_stats_df['file_size_kb'].min():.1f} - {image_stats_df['file_size_kb'].max():.1f} KB")

print(f"\nAspect Ratios:")
print(f"  • Mean: {image_stats_df['aspect_ratio'].mean():.2f}")
print(f"  • Std: {image_stats_df['aspect_ratio'].std():.2f}")
print(f"  • Range: {image_stats_df['aspect_ratio'].min():.2f} - {image_stats_df['aspect_ratio'].max():.2f}")

print(f"\nColor Channels (Mean values 0-255):")
print(f"  • Red   - Mean: {image_stats_df['r_mean'].mean():.1f}, Std: {image_stats_df['r_mean'].std():.1f}")
print(f"  • Green - Mean: {image_stats_df['g_mean'].mean():.1f}, Std: {image_stats_df['g_mean'].std():.1f}")
print(f"  • Blue  - Mean: {image_stats_df['b_mean'].mean():.1f}, Std: {image_stats_df['b_mean'].std():.1f}")

print(f"\nQuality Metrics:")
print(f"  • Brightness - Mean: {image_stats_df['brightness'].mean():.1f}, Range: {image_stats_df['brightness'].min():.1f} - {image_stats_df['brightness'].max():.1f}")
print(f"  • Contrast - Mean: {image_stats_df['contrast'].mean():.1f}")
print(f"  • Sharpness - Mean: {image_stats_df['sharpness'].mean():.1f}")

Total images to analyze: 46229

STEP 1: Collecting Image Metadata...


Scanning images: 100%|██████████| 46229/46229 [10:18<00:00, 74.74it/s]  



✓ Successfully analyzed 42000 images
⚠ Failed to analyze 4229 images

IMAGE STATISTICS SUMMARY

Dimensions:
  • Width  - Mean: 100px, Std: 0px
  • Height - Mean: 100px, Std: 0px
  • Range: 100×100 to 100×100

File Sizes:
  • Mean: 5.2 KB
  • Median: 5.1 KB
  • Range: 0.4 - 13.5 KB

Aspect Ratios:
  • Mean: 1.00
  • Std: 0.00
  • Range: 1.00 - 1.00

Color Channels (Mean values 0-255):
  • Red   - Mean: 192.3, Std: 40.6
  • Green - Mean: 190.1, Std: 41.0
  • Blue  - Mean: 184.7, Std: 43.9

Quality Metrics:
  • Brightness - Mean: 189.0, Range: 9.8 - 254.0
  • Contrast - Mean: 68.3
  • Sharpness - Mean: 0.0


In [11]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Bật chế độ mở biểu đồ trên tab trình duyệt mới (tránh lỗi nbformat)
pio.renderers.default = "browser" 
pio.templates.default = "plotly_white"

# Lấy ra Top 5 danh mục có nhiều ảnh nhất
top_n = 5
top_categories = image_stats_df['category'].value_counts().head(top_n).index.tolist()

# Tạo một dataframe mới chỉ chứa ảnh của Top 5 danh mục này
df_top_cats = image_stats_df[image_stats_df['category'].isin(top_categories)].copy()

print(f"Tổng số ảnh ban đầu: {len(image_stats_df)}")
print(f"Đã lọc ra {len(df_top_cats)} ảnh thuộc Top {top_n} danh mục để biểu đồ không bị rối.")
print(f"Các danh mục gồm: {', '.join(top_categories)}")

Tổng số ảnh ban đầu: 42000
Đã lọc ra 10000 ảnh thuộc Top 5 danh mục để biểu đồ không bị rối.
Các danh mục gồm: Arts, Crafts & Sewing, Cell Phones & Accessories, Clothing, Shoes & Jewelry, Tools & Home Improvement, Health & Personal Care


In [13]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Bật chế độ mở trên trình duyệt
pio.renderers.default = "browser" 
pio.templates.default = "plotly_white"

print(f"Sẵn sàng vẽ biểu đồ cho toàn bộ {len(image_stats_df)} ảnh với tất cả các danh mục!")

Sẵn sàng vẽ biểu đồ cho toàn bộ 42000 ảnh với tất cả các danh mục!


In [14]:
fig_size = px.scatter(
    image_stats_df,  # Dùng toàn bộ dataset
    x="width", 
    y="height",
    color="category",
    marginal_x="histogram", 
    marginal_y="histogram",
    title="Image Size Distribution (Width x Height) - All Categories",
    labels={"width": "Width (pixels)", "height": "Height (pixels)"},
    opacity=0.6
)
# Vì có nhiều danh mục, lề phải (chứa legend) có thể sẽ bị dài, Plotly sẽ tự cho phép cuộn (scroll)
fig_size.show()

In [7]:
import sys
!{sys.executable} -m pip install --upgrade nbformat ipython

  Using cached ipython-9.12.0-py3-none-any.whl.metadata (4.5 kB)
Using cached ipython-9.12.0-py3-none-any.whl (625 kB)
  Attempting uninstall: ipython
    Found existing installation: ipython 9.8.0
    Uninstalling ipython-9.8.0:
      Successfully uninstalled ipython-9.8.0


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [15]:
# 1. Biểu đồ dung lượng file
fig_filesize = px.histogram(
    image_stats_df,
    x="file_size_kb",
    color="category",
    title="File Size Distribution - All Categories",
    labels={"file_size_kb": "File Size (KB)"},
    barmode="overlay",
    opacity=0.5, # Giảm opacity xuống một chút để nhìn xuyên qua các lớp màu tốt hơn
    nbins=100
)
fig_filesize.show()

# 2. Biểu đồ tỷ lệ khung hình
fig_aspect = px.histogram(
    image_stats_df,
    x="aspect_ratio",
    color="category",
    title="Aspect Ratio Distribution - All Categories",
    labels={"aspect_ratio": "Aspect Ratio (Width/Height)"},
    barmode="overlay",
    opacity=0.5,
    nbins=100
)

# Thêm các vạch đỏ đánh dấu
common_ratios = {0.75: '0.75:1', 1.0: '1.0:1', 1.33: '1.33:1', 1.5: '1.5:1', 2.0: '2.0:1'}
for ratio, label in common_ratios.items():
    fig_aspect.add_vline(
        x=ratio, line_dash="dash", line_color="red", 
        annotation_text=label, annotation_position="top left"
    )

fig_aspect.update_xaxes(range=[0.4, 2.7]) 
fig_aspect.show()

In [17]:
import cv2
import numpy as np
from tqdm import tqdm

print("Đang tính toán lại Sharpness bằng OpenCV...")

new_sharpness = []

# Duyệt qua các ID ảnh đã có trong dataframe
for img_id in tqdm(image_stats_df['ImgId']):
    img_path = f"data/train/train/{img_id}.jpg"
    try:
        # Đọc ảnh trực tiếp ở dạng Trắng Đen (Grayscale) bằng OpenCV
        img_cv = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # Tính Laplacian Variance chuẩn (dùng CV_64F để không bị mất giá trị âm)
        variance = cv2.Laplacian(img_cv, cv2.CV_64F).var()
        new_sharpness.append(variance)
    except:
        new_sharpness.append(0) # Đề phòng có ảnh bị lỗi không đọc được

# Cập nhật lại cột sharpness trong DataFrame hiện tại
image_stats_df['sharpness'] = new_sharpness

print("Hoàn tất! Vẽ lại biểu đồ...")

# Vẽ lại biểu đồ Quality Metrics
import plotly.express as px

fig_quality = px.scatter(
    image_stats_df,
    x="sharpness", 
    y="contrast",
    color="category",
    title="Image Quality Metrics - All Categories (Fixed)",
    labels={
        "sharpness": "Sharpness (Laplacian Variance - OpenCV)", 
        "contrast": "Contrast (Standard Deviation)"
    },
    opacity=0.6
)

# Thường Sharpness tính bằng cv2 sẽ có dải giá trị từ 0 đến vài nghìn. 
# Nếu có vài ảnh quá nhiễu làm trục x kéo dài tít tắp, bạn có thể uncomment dòng dưới để giới hạn lại trục X
# fig_quality.update_xaxes(range=[0, 15000])

fig_quality.show()

Đang tính toán lại Sharpness bằng OpenCV...


100%|██████████| 42000/42000 [04:05<00:00, 171.26it/s]


Hoàn tất! Vẽ lại biểu đồ...


In [18]:
import plotly.express as px

# Cell 4: RGB Color Space Distribution (Biểu đồ 3D) - Bản Fix lỗi mất lề đáy
fig_rgb = px.scatter_3d(
    image_stats_df, 
    x='r_mean', 
    y='g_mean', 
    z='b_mean',
    color='brightness',
    hover_name='category', 
    title="RGB Color Space Distribution (All Products)",
    labels={'r_mean': 'Red', 'g_mean': 'Green', 'b_mean': 'Blue'},
    opacity=0.8,
    color_continuous_scale='Viridis'
)

# Chỉnh hạt nhỏ lại
fig_rgb.update_traces(marker=dict(size=3)) 

# ĐÃ SỬA: b=50 để tạo không gian cho nhãn trục dưới cùng không bị cắt lẹm
fig_rgb.update_layout(margin=dict(l=0, r=0, b=50, t=40))

fig_rgb.show()

In [19]:
import os

# Tạo thư mục để chứa file JSON nếu chưa có
os.makedirs("assets/data/charts", exist_ok=True)

# Xuất các biểu đồ ra file JSON
fig_size.write_json("assets/data/charts/size_marginal.json")
fig_filesize.write_json("assets/data/charts/file_size.json")
fig_aspect.write_json("assets/data/charts/aspect_ratio.json")
fig_rgb.write_json("assets/data/charts/color_space.json")
fig_quality.write_json("assets/data/charts/quality_metrics.json")

print("Đã xuất xong các file JSON!")

Đã xuất xong các file JSON!


In [25]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import os

pio.renderers.default = "browser"
os.makedirs("assets/data/charts", exist_ok=True)

# 1. Load dữ liệu
train_df = pd.read_csv('data/train/train.csv')
category_counts = train_df['categories'].value_counts()

# Chuyển đổi toàn bộ category_counts thành DataFrame
all_cats = category_counts.reset_index()
all_cats.columns = ['categories', 'Count']

# --- Biểu đồ 1: Bar Chart TẤT CẢ Danh mục ---
fig_class_bar = px.bar(
    all_cats, 
    x='categories', 
    y='Count',
    title=f"All {len(all_cats)} Product Categories Distribution",
    color='Count',
    color_continuous_scale='Purples'
)
fig_class_bar.update_layout(
    xaxis_tickangle=-45,
    margin=dict(b=150) # Tăng lề dưới để đủ chỗ hiển thị tên tất cả các class
)
fig_class_bar.write_json("assets/data/charts/class_dist_bar.json")
fig_class_bar.show()

# --- Biểu đồ 2: Pie Chart TẤT CẢ Danh mục ---
fig_species_pie = px.pie(
    all_cats, 
    names='categories', 
    values='Count',
    title=f"Category Distribution (All {len(all_cats)} Classes)",
    hole=0.4, # Tạo dáng Donut chart cho hiện đại
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# Ẩn bớt text dán trực tiếp lên các lát cắt quá nhỏ để đỡ rối mắt
fig_species_pie.update_traces(textposition='inside', textinfo='percent')

fig_species_pie.write_json("assets/data/charts/species_pie.json")
fig_species_pie.show()

In [26]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
from tqdm import tqdm

print("Khởi tạo mô hình ResNet50 pretrained...")
# Tải mô hình ResNet50, loại bỏ lớp phân loại cuối cùng để lấy vector đặc trưng
weights = models.ResNet50_Weights.DEFAULT
resnet = models.resnet50(weights=weights)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()

# Nếu có GPU thì đẩy model lên GPU cho nhanh
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--> Đang chạy trên thiết bị: {device}")
resnet = resnet.to(device)

# Tiền xử lý ảnh chuẩn của ResNet
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

features = []
valid_labels = []

print(f"Bắt đầu trích xuất đặc trưng cho TOÀN BỘ {len(train_df)} ảnh...")
print("(Có thể mất vài phút đến vài chục phút tùy cấu hình máy...)")

with torch.no_grad():
    # Chạy vòng lặp qua toàn bộ train_df thay vì sample_df
    for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
        img_path = f"data/train/train/{row['ImgId']}.jpg"
        try:
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0).to(device)
            feat = resnet(img_tensor).squeeze().cpu().numpy()
            features.append(feat)
            valid_labels.append(row['category'])
        except Exception as e:
            continue

X = np.array(features)
y = np.array(valid_labels)
print(f"\nTrích xuất thành công: Ma trận đặc trưng có kích thước {X.shape}")

Khởi tạo mô hình ResNet50 pretrained...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\LENOVO/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [03:23<00:00, 503kB/s] 


--> Đang chạy trên thiết bị: cpu
Bắt đầu trích xuất đặc trưng cho TOÀN BỘ 46229 ảnh...
(Có thể mất vài phút đến vài chục phút tùy cấu hình máy...)


100%|██████████| 46229/46229 [55:29<00:00, 13.88it/s]  



Trích xuất thành công: Ma trận đặc trưng có kích thước (42000, 2048)


In [39]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap
import numpy as np
import plotly.express as px
import pandas as pd

print("0. Đang lấy mẫu (Sampling) 50 mẫu mỗi Class...")

# Bước 0.1: Đảm bảo y và X khớp nhau trước (xử lý lỗi 46k vs 42k cũ)
y_full = train_df['categories'].iloc[:X.shape[0]].reset_index(drop=True)

# Bước 0.2: Thuật toán chọn 50 index mỗi class
sampled_indices = []
for label in y_full.unique():
    # Lấy tất cả index của class hiện tại
    idx_list = y_full[y_full == label].index.tolist()
    
    # Lấy ngẫu nhiên 50 cái (hoặc lấy hết nếu class đó ít hơn 50)
    n_samples = min(50, len(idx_list))
    selected_idx = np.random.choice(idx_list, size=n_samples, replace=False)
    sampled_indices.extend(selected_idx)

# Bước 0.3: Trích xuất tập dữ liệu nhỏ (Subset)
X_subset = X[sampled_indices]
y_subset = y_full.iloc[sampled_indices]

print(f"✅ Đã chọn xong: {X_subset.shape[0]} mẫu cho {len(y_full.unique())} classes.")

# --- 1. PCA ---
print("1. Tính toán PCA trên tập mẫu...")
pca_full = PCA(n_components=50)
X_pca_subset = pca_full.fit_transform(X_subset) # Fit và transform luôn trên tập nhỏ

# --- 2. t-SNE ---
print("2. Chạy t-SNE (Lần này sẽ rất nhanh vì chỉ có ~1000 điểm)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)
X_tsne = tsne.fit_transform(X_pca_subset)

fig_tsne = px.scatter(
    x=X_tsne[:, 0], 
    y=X_tsne[:, 1], 
    color=y_subset, 
    title="t-SNE: 50 samples per Class (Visual Balance)",
    labels={'color': 'Categories', 'x': 't-SNE 1', 'y': 't-SNE 2'},
    opacity=0.8,
    hover_data=[y_subset] # Cho phép xem tên class khi di chuột
)
fig_tsne.update_traces(marker=dict(size=6, line=dict(width=1, color='DarkSlateGrey'))) # Điểm to hơn cho dễ nhìn
fig_tsne.write_json("assets/data/charts/tsne_sampled_plot.json")
fig_tsne.show()

# --- 3. UMAP ---
print("3. Chạy UMAP...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42, n_jobs=-1)
X_umap = reducer.fit_transform(X_subset)

fig_umap = px.scatter(
    x=X_umap[:, 0], 
    y=X_umap[:, 1], 
    color=y_subset,
    title="UMAP: 50 samples per Class (Visual Balance)",
    labels={'color': 'Categories', 'x': 'UMAP 1', 'y': 'UMAP 2'},
    opacity=0.8
)
fig_umap.update_traces(marker=dict(size=6, line=dict(width=1, color='DarkSlateGrey')))
fig_umap.write_json("assets/data/charts/umap_sampled_plot.json")
fig_umap.show()

print("🚀 HOÀN TẤT! Biểu đồ trông sẽ rất sạch sẽ và chuyên nghiệp.")

0. Đang lấy mẫu (Sampling) 50 mẫu mỗi Class...
✅ Đã chọn xong: 1050 mẫu cho 21 classes.
1. Tính toán PCA trên tập mẫu...
2. Chạy t-SNE (Lần này sẽ rất nhanh vì chỉ có ~1000 điểm)...
3. Chạy UMAP...
🚀 HOÀN TẤT! Biểu đồ trông sẽ rất sạch sẽ và chuyên nghiệp.


In [36]:
fig_tsne.show()

In [37]:
fig_umap.show()

In [40]:
import pandas as pd
import plotly.express as px
from sklearn.metrics.pairwise import cosine_similarity

print("4. Đang tính toán Ma trận Tương đồng (Cosine Similarity)...")
# Gom nhóm theo category và tính giá trị trung bình của vector đặc trưng
df_features = pd.DataFrame(X)
df_features['category'] = y
mean_features = df_features.groupby('category').mean()

# Tính cosine similarity giữa tất cả các danh mục
labels = mean_features.index.tolist()
sim_matrix = cosine_similarity(mean_features.values)

print(f"Đang vẽ biểu đồ Heatmap cho toàn bộ {len(labels)} danh mục...")
fig_sim = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    title=f"Category Similarity Matrix (Cosine) - All {len(labels)} Classes",
    color_continuous_scale='Viridis',
    labels=dict(color="Similarity")
)

# Chỉnh sửa layout để hiển thị đẹp khi có quá nhiều class
fig_sim.update_layout(
    xaxis_tickangle=-45,
    margin=dict(l=150, r=50, t=60, b=150), # Nới rộng lề trái (l) và dưới (b) để tên category không bị cắt
    width=1000, height=1000 # Phóng to toàn bộ biểu đồ
)

# Xuất ra JSON để đưa lên HTML
fig_sim.write_json("assets/data/charts/similarity_heatmap.json")

# Hiển thị trên trình duyệt
fig_sim.show()

print("\n🎉 HOÀN TẤT TOÀN BỘ QUÁ TRÌNH EDA!")
print("Tất cả các file JSON đã được lưu an toàn vào thư mục: assets/data/charts/")

4. Đang tính toán Ma trận Tương đồng (Cosine Similarity)...
Đang vẽ biểu đồ Heatmap cho toàn bộ 21 danh mục...

🎉 HOÀN TẤT TOÀN BỘ QUÁ TRÌNH EDA!
Tất cả các file JSON đã được lưu an toàn vào thư mục: assets/data/charts/


In [ ]:
loadPlotlyChart('assets/data/charts/class_dist_bar.json', 'class-dist-bar');
loadPlotlyChart('assets/data/charts/species_pie.json', 'species-pie');
loadPlotlyChart('assets/data/charts/tsne_plot.json', 'tsne-plot');
loadPlotlyChart('assets/data/charts/umap_plot.json', 'umap-plot');
loadPlotlyChart('assets/data/charts/pca_variance.json', 'pca-variance');
loadPlotlyChart('assets/data/charts/similarity_heatmap.json', 'similarity-heatmap');

In [51]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import re
import os

# 0. Storage Configuration
output_dir = "assets/data/charts"
os.makedirs(output_dir, exist_ok=True)

# 1. Load Data
train_csv_path = "data/train/train.csv"
df = pd.read_csv(train_csv_path)

# 2. Pre-processing (Column Mapping)
# Combining title and description for a comprehensive text analysis
df['text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')
df['label'] = df['categories'] 

# 3. Setup Stop Words
stop_words = set(ENGLISH_STOP_WORDS)
# Adding domain-specific stop words for E-commerce/Retail
custom_retail_stopwords = ['cm', 'mm', 'kg', 'size', 'color', 'new', 'used', 'product', 'item', 'black', 'white']
stop_words.update(custom_retail_stopwords)

print(f"✅ Total Products Loaded: {len(df)}")

# --- CHART 1: WORD COUNT DISTRIBUTION ---
print("📊 Processing Word Count Distribution...")
df['word_count'] = df['text'].str.split().str.len()

fig_dist = go.Figure(data=[go.Histogram(
    x=df['word_count'],
    nbinsx=50,
    marker_color='#667eea',
    opacity=0.75
)])

fig_dist.update_layout(
    title='Word Count Distribution per Product',
    xaxis_title='Number of Words',
    yaxis_title='Number of Products',
    template="plotly_white"
)

# Save and Display Chart 1
fig_dist.write_json(os.path.join(output_dir, "word_count_dist.json"))
fig_dist.show()


# --- CHART 2: TOP 20 KEYWORDS ---
print("📊 Processing Top 20 Keywords...")

# Extracting keywords from the first 42,000 rows to maintain consistency
all_text = ' '.join(df['text'].iloc[:42000]).lower()
words = re.findall(r'\b[a-z]{3,}\b', all_text)
meaningful_words = [w for w in words if w not in stop_words]

top_20 = Counter(meaningful_words).most_common(20)
words_list, counts_list = zip(*top_20)

fig_words = go.Figure(data=[go.Bar(
    x=words_list,
    y=counts_list,
    marker_color='#43e97b',
    text=counts_list,
    textposition='outside'
)])

fig_words.update_layout(
    title='Top 20 Product Keywords (Stop Words Removed)',
    xaxis_title='Keywords',
    yaxis_title='Frequency',
    xaxis={'tickangle': 45},
    template="plotly_white"
)

# Save and Display Chart 2
fig_words.write_json(os.path.join(output_dir, "top_keywords.json"))
fig_words.show()

print(f"🚀 COMPLETED! Charts have been saved to: {output_dir}")

✅ Total Products Loaded: 46229
📊 Processing Word Count Distribution...
📊 Processing Top 20 Keywords...
🚀 COMPLETED! Charts have been saved to: assets/data/charts


In [56]:
import nltk
from nltk.corpus import stopwords
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import os

# 1. Download NLTK data
nltk.download('stopwords')

# 2. Setup directory
output_dir = "assets/data/charts"
os.makedirs(output_dir, exist_ok=True)

# 3. Define NLTK Stopwords + Custom Retail Noise
base_stop_words = set(stopwords.words('english'))
custom_retail_stopwords = {'cm', 'mm', 'kg', 'size', 'color', 'new', 'used', 'product', 'item', 'black', 'white'}
stop_words = base_stop_words.union(custom_retail_stopwords)

print(f"✅ NLTK Setup Complete. Using {len(stop_words)} stop words.")

✅ NLTK Setup Complete. Using 209 stop words.


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [57]:
# Load dataset
train_df = pd.read_csv("data/train/train.csv")

# Combine title and description
train_df['text'] = train_df['title'].fillna('') + ' ' + train_df['description'].fillna('')

# Align to your 42,000 samples
y = train_df['categories'].iloc[:42000].reset_index(drop=True)
text_data = train_df['text'].iloc[:42000].reset_index(drop=True)

print(f"✅ Data Ready: {len(text_data)} samples aligned with labels.")

✅ Data Ready: 42000 samples aligned with labels.


In [59]:
print("📊 1/6: Stop Words Analysis...")
all_tokens = ' '.join(text_data).lower().split()
stop_counts = Counter([w for w in all_tokens if w in stop_words])
top_stops = stop_counts.most_common(20)
s_words, s_counts = zip(*top_stops)

fig1 = go.Figure(data=[go.Bar(x=s_words, y=s_counts, marker_color='#764ba2', text=s_counts, textposition='outside')])
fig1.update_layout(title='Top 20 NLTK Stop Words Found', xaxis_title='Stop Words', yaxis_title='Frequency', template="plotly_white")

fig1.write_json(os.path.join(output_dir, "stop_words.json"))
fig1.show()

📊 1/6: Stop Words Analysis...


In [60]:
print("📊 2/6: Vocabulary Richness...")
vocab_stats = []
for category in sorted(y.unique()):
    cat_text = ' '.join(text_data[y == category]).lower()
    words = re.findall(r'\b[a-z]{3,}\b', cat_text)
    meaningful = [w for w in words if w not in stop_words]
    
    unique_cnt = len(set(meaningful))
    total_cnt = len(meaningful)
    ttr = unique_cnt / total_cnt if total_cnt > 0 else 0
    vocab_stats.append({'category': category, 'unique_words': unique_cnt})

v_df = pd.DataFrame(vocab_stats)
fig2 = go.Figure(data=[go.Bar(x=v_df['category'], y=v_df['unique_words'], marker_color='#4facfe')])
fig2.update_layout(title='Unique Vocabulary per Category', xaxis_title='Category', yaxis_title='Count', template="plotly_white")

fig2.write_json(os.path.join(output_dir, "vocab_richness.json"))
fig2.show()

📊 2/6: Vocabulary Richness...


In [62]:
# --- STEP A: TF-IDF Calculation (Run once) ---
vectorizer = TfidfVectorizer(max_features=5000, stop_words=list(stop_words))
tfidf_matrix = vectorizer.fit_transform(text_data)
feature_names = vectorizer.get_feature_names_out()

# --- STEP B: Loop through ALL Categories ---
all_categories = sorted(y.unique())
print(f"🚀 Starting deep dive for {len(all_categories)} categories...")

for category in all_categories:
    print(f"📊 Processing: {category}...")
    
    # 1. Lọc index của category hiện tại
    cat_indices = y[y == category].index
    
    # 2. Tính Mean TF-IDF scores cho category này
    mean_tfidf = np.array(tfidf_matrix[cat_indices].mean(axis=0)).flatten()
    
    # 3. Lấy Top 20 terms
    top_tfidf_idx = mean_tfidf.argsort()[-20:][::-1]
    t_words = [feature_names[i] for i in top_tfidf_idx]
    t_scores = [mean_tfidf[i] for i in top_tfidf_idx]
    
    # 4. Khởi tạo biểu đồ
    fig = go.Figure(data=[go.Bar(
        x=t_scores, 
        y=t_words, 
        orientation='h', 
        marker_color='#764ba2'
    )])
    
    fig.update_layout(
        title=f'Distinctive Terms (TF-IDF): {category}',
        xaxis_title='Mean TF-IDF Score',
        yaxis_title='Terms',
        yaxis={'autorange': 'reversed'},
        template="plotly_white",
        width=800,
        height=600
    )
    
    # 5. Lưu file JSON riêng cho từng class
    # Làm sạch tên category để làm tên file (xóa khoảng trắng, ký tự đặc biệt)
    safe_name = re.sub(r'[^a-zA-Z0-9]', '_', category).lower()
    file_path = os.path.join(output_dir, f"tfidf_{safe_name}.json")
    
    fig.write_json(file_path)
    
    # Tùy chọn: Chỉ show biểu đồ của 3 class đầu tiên để tránh làm treo trình duyệt
    if category in all_categories[:3]:
        fig.show()

print(f"✅ Hoàn tất! Đã lưu {len(all_categories)} file JSON vào thư mục {output_dir}")

🚀 Starting deep dive for 21 categories...
📊 Processing: All Beauty...
📊 Processing: All Electronics...
📊 Processing: Appliances...
📊 Processing: Arts, Crafts & Sewing...
📊 Processing: Automotive...
📊 Processing: Baby...
📊 Processing: Baby Products...
📊 Processing: Beauty...
📊 Processing: Cell Phones & Accessories...
📊 Processing: Clothing, Shoes & Jewelry...
📊 Processing: Electronics...
📊 Processing: Grocery & Gourmet Food...
📊 Processing: Health & Personal Care...
📊 Processing: Industrial & Scientific...
📊 Processing: Musical Instruments...
📊 Processing: Office Products...
📊 Processing: Patio, Lawn & Garden...
📊 Processing: Pet Supplies...
📊 Processing: Sports & Outdoors...
📊 Processing: Tools & Home Improvement...
📊 Processing: Toys & Games...
✅ Hoàn tất! Đã lưu 21 file JSON vào thư mục assets/data/charts


In [65]:
# --- STEP A: Setup N-gram Vectorizer ---
# We use TF-IDF weighting on bigrams to find meaningful phrases 
# that are unique to each specific retail category.

all_categories = sorted(y.unique())
print(f"🚀 Starting Bigram extraction for {len(all_categories)} categories...")

for category in all_categories:
    print(f"🔗 Extracting Bigrams for: {category}...")
    
    # 1. Combine all text for the current category into one large string
    combined_cat_text = ' '.join(text_data[y == category].values)
    
    # 2. Extract Bigrams (2-word phrases) using TF-IDF
    bigram_vec = TfidfVectorizer(
        ngram_range=(2, 2), # Focus on pairs of words
        max_features=50,
        stop_words=list(stop_words)
    )
    
    # Fit on the single combined text block
    bigram_mtx = bigram_vec.fit_transform([combined_cat_text])
    b_names = bigram_vec.get_feature_names_out()
    b_scores = bigram_mtx.toarray()[0]
    
    # 3. Sort and get Top 15 bigrams
    top_b_indices = b_scores.argsort()[-15:][::-1]
    top_bigrams = [b_names[i] for i in top_b_indices]
    top_b_scores = [b_scores[i] for i in top_b_indices]
    
    # 4. Create Visualization
    fig = go.Figure(data=[go.Bar(
        x=top_b_scores, 
        y=top_bigrams, 
        orientation='h', 
        marker_color='#4facfe',
        text=[f'{s:.4f}' for s in top_b_scores],
        textposition='outside'
    )])
    
    fig.update_layout(
        title=f'Top 15 Bigrams in {category}',
        xaxis_title='TF-IDF Score',
        yaxis_title='Bigram Phrases',
        yaxis={'autorange': 'reversed'},
        template="plotly_white",
        width=900,
        height=600
    )
    
    # 5. Save JSON for the dashboard
    safe_name = re.sub(r'[^a-zA-Z0-9]', '_', category).lower()
    file_path = os.path.join(output_dir, f"bigrams_{safe_name}.json")
    fig.write_json(file_path)
    
    # Display the first 3 classes to check results
    if category in all_categories[:3]:
        fig.show()

print(f"✅ Success! Bigram JSON files saved in: {output_dir}")

🚀 Starting Bigram extraction for 21 categories...
🔗 Extracting Bigrams for: All Beauty...
🔗 Extracting Bigrams for: All Electronics...
🔗 Extracting Bigrams for: Appliances...
🔗 Extracting Bigrams for: Arts, Crafts & Sewing...
🔗 Extracting Bigrams for: Automotive...
🔗 Extracting Bigrams for: Baby...
🔗 Extracting Bigrams for: Baby Products...
🔗 Extracting Bigrams for: Beauty...
🔗 Extracting Bigrams for: Cell Phones & Accessories...
🔗 Extracting Bigrams for: Clothing, Shoes & Jewelry...
🔗 Extracting Bigrams for: Electronics...
🔗 Extracting Bigrams for: Grocery & Gourmet Food...
🔗 Extracting Bigrams for: Health & Personal Care...
🔗 Extracting Bigrams for: Industrial & Scientific...
🔗 Extracting Bigrams for: Musical Instruments...
🔗 Extracting Bigrams for: Office Products...
🔗 Extracting Bigrams for: Patio, Lawn & Garden...
🔗 Extracting Bigrams for: Pet Supplies...
🔗 Extracting Bigrams for: Sports & Outdoors...
🔗 Extracting Bigrams for: Tools & Home Improvement...
🔗 Extracting Bigrams for: 

In [1]:
print("📊 6/6: Generating High-Resolution Category Similarity Matrix...")

# 1. Identify all unique categories
categories = sorted(y.unique())
n_cats = len(categories)
similarity_matrix = np.zeros((n_cats, n_cats))

# 2. Compute Mean Pairwise Similarity
for i, cat1 in enumerate(categories):
    cat1_indices = y[y == cat1].index
    cat1_vectors = tfidf_matrix[cat1_indices]
    
    for j, cat2 in enumerate(categories):
        if i <= j:
            cat2_indices = y[y == cat2].index
            cat2_vectors = tfidf_matrix[cat2_indices]
            
            pairwise_sim = cosine_similarity(cat1_vectors, cat2_vectors)
            mean_sim = np.mean(pairwise_sim)
            
            similarity_matrix[i, j] = mean_sim
            similarity_matrix[j, i] = mean_sim

# 3. Create Large Interactive Heatmap
fig6 = go.Figure(data=go.Heatmap(
    z=similarity_matrix,
    x=categories,
    y=categories,
    colorscale='Purples',
    text=np.round(similarity_matrix, 3),
    texttemplate='%{text}',
    # Increased text size for the numbers inside cells
    textfont={"size": 11}, 
    colorbar=dict(
        title="Similarity Score",
        thickness=20,
        len=0.8
    )
))

fig6.update_layout(
    title='Category Similarity Matrix (Cosine Similarity) - All 21 Classes',
    xaxis_title='Predicted Category',
    yaxis_title='True Category',
    # INCREASED SIZE HERE:
    width=1200, 
    height=1200,
    # Adjusting margins to prevent label clipping
    margin=dict(l=150, r=50, b=150, t=100),
    xaxis=dict(tickfont=dict(size=12), tickangle=45),
    yaxis=dict(tickfont=dict(size=12), autorange='reversed'),
    template="plotly_white"
)

# 4. Save and Show
print(f"💾 Saving 1200x1200px matrix to JSON...")
fig6.write_json(os.path.join(output_dir, "similarity_matrix_large.json"))

print("📊 Rendering Large Heatmap (Opening in new tab/cell)...")
fig6.show()

📊 6/6: Generating High-Resolution Category Similarity Matrix...


NameError: name 'y' is not defined

In [68]:
import numpy as np
import plotly.graph_objects as go
import os

print("📊 Analyzing Word Count Distribution...")

# 1. Calculate word counts for the aligned 42,000 samples
# We use the 'text_data' variable defined in Cell 2
word_counts = text_data.str.split().str.len()

# 2. Create histogram bins (intervals of 50 words for better granularity)
max_val = int(word_counts.max())
bins = np.arange(0, max_val + 50, 50)
hist, bin_edges = np.histogram(word_counts, bins=bins)

# Format bin labels for the X-axis
bin_labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(hist))]

# 3. Create the Visualization
fig_dist = go.Figure(data=[go.Bar(
    x=bin_labels,
    y=hist,
    marker_color='#667eea',
    text=hist,
    textposition='outside'
)])

fig_dist.update_layout(
    title='Product Word Count Distribution (First 42,000 Samples)',
    xaxis_title='Word Count Ranges',
    yaxis_title='Number of Products',
    width=1000,
    height=500,
    xaxis={'tickangle': 45},
    template="plotly_white"
)

# 4. Save and Show
print(f"💾 Saving distribution chart to JSON...")
fig_dist.write_json(os.path.join(output_dir, "word_count_distribution.json"))

print("📊 Rendering Distribution Chart...")
fig_dist.show()

# 5. Print Detailed Statistics for Model Configuration
print("\n--- Word Count Statistics ---")
print(f"Mean (Average): {word_counts.mean():.2f}")
print(f"Median:         {word_counts.median():.2f}")
print(f"Minimum:        {word_counts.min()}")
print(f"Maximum:        {word_counts.max()}")

print("\n--- Percentiles (Important for Padding/Truncation) ---")
percentiles = [25, 50, 75, 90, 95, 99]
for p in percentiles:
    val = word_counts.quantile(p/100)
    print(f"{p}th Percentile: {val:.0f} words")

📊 Analyzing Word Count Distribution...
💾 Saving distribution chart to JSON...
📊 Rendering Distribution Chart...

--- Word Count Statistics ---
Mean (Average): 82.52
Median:         57.00
Minimum:        1
Maximum:        1958

--- Percentiles (Important for Padding/Truncation) ---
25th Percentile: 30 words
50th Percentile: 57 words
75th Percentile: 102 words
90th Percentile: 171 words
95th Percentile: 237 words
99th Percentile: 447 words
